# Task 1.2 — Key Assumptions

**Selected Paper:** Learning Kernels with Radiuses of Minimum Enclosing Balls (NeurIPS 2010)

This notebook identifies three assumptions that the Radius-based Kernel Learning (RKL) method relies on, explains why each is needed, and describes what happens when it is violated.

---
### Assumption 1
**The kernel matrix $K(\theta)$ must be strictly positive definite**

**Assumption:** The combined kernel matrix $K(\theta) = \sum_j \theta_j K_j$ is strictly positive definite for all feasible $\theta$.

**Why the method needs it:** The differentiability of the top-level objective $g(\theta)$ is established via Theorem 1 (Section 4), which requires that both the SVM dual problem (Eq. 16) and the MEB dual problem (Eq. 17) have **unique** optimal solutions $\alpha^*$ and $\beta^*$. Strict positive definiteness of $K(\theta)$ guarantees this uniqueness, since both duals are then strictly concave quadratic programs. Without it, the gradient $\nabla_\theta g$ (Eq. 21) used in the gradient projection algorithm (Section 5) is not well-defined.

**Violation scenario:** If the basis kernels are chosen such that their combination $K(\theta)$ becomes only positive semi-definite (e.g., a linear kernel on data that lies in a lower-dimensional subspace), the SVM and MEB duals may have multiple optimal solutions. In this case, $g(\theta)$ may only be directionally differentiable rather than continuously differentiable, and the gradient projection algorithm can fail to converge or produce inconsistent gradient estimates.

**Paper reference:** Section 4, paragraph following Theorem 1: *"This can be guaranteed by that the kernel matrix $K(\theta)$ is strictly positive definite."*

---
### Assumption 2
**The linear combination of basis kernels is expressive enough for the task**

**Assumption:** The kernel family $\mathcal{K} = \{k^{(\theta)} = \sum_j \theta_j k_j \mid \theta_j \geq 0\}$ contains a kernel that is well-suited to the classification problem at hand.

**Why the method needs it:** The RKL formulation (Eq. 9) searches for the best kernel within the prescribed family $\mathcal{K}$. If none of the basis kernels $k_j$ — nor any nonnegative combination of them — captures the relevant structure of the data, the method can only return the best among poor choices. The margin-radius ratio $R^2(k)\|w\|^2$ correctly ranks kernels within $\mathcal{K}$, but it cannot compensate for an inadequate family of kernels.

**Violation scenario:** Suppose the true decision boundary requires a specific non-stationary kernel, but the basis set consists only of Gaussian RBF kernels with narrow bandwidths. No linear combination of these basis kernels will produce a good feature space for the task, and RKL will return a kernel with poor generalisation despite correctly optimising the margin-radius objective.

**Paper reference:** Section 2, Eq. 2 and Section 3: the formulation optimises over $k^{(\theta)} = \sum_j \theta_j k_j$ with $\theta_j \geq 0$; the quality of the solution is inherently bounded by the expressiveness of this family.

---
### Assumption 3
**The MEB radius meaningfully represents data spread in feature space**

**Assumption:** The radius of the Minimum Enclosing Ball (MEB) meaningfully represents the spread of the dataset in feature space.

**Why the method needs it:** The RKL objective (Eq. 9) multiplies the margin term with $R^2(k)$. If the radius does not accurately represent the data spread, the optimization objective will not correctly reflect generalization behaviour. The entire formulation relies on $R^2(k)$ being a faithful summary of how the data is distributed in the feature space induced by the learned kernel.

**Violation scenario:** Datasets containing extreme outliers can artificially increase the radius of the minimum enclosing ball, which can distort the optimization objective and lead to poor kernel learning. In such cases, a single outlier mapped far from the rest of the data in feature space would inflate $R^2(k)$, causing the method to prefer kernels that shrink the feature space rather than kernels that genuinely separate the classes.

**Paper reference:** Section 2.2 and Equations (7–8) in the paper.